In [ ]:
!pip -q install essentia tensorflow

In [ ]:
import essentia
import essentia.standard as es

print(essentia.__version__)

2.1-beta6-dev


In [ ]:
import os
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

import essentia
import essentia.standard as es


In [ ]:
import essentia
import essentia.standard as es

print(essentia.__version__)
print(hasattr(es, "TensorflowPredictMusiCNN"))

2.1-beta6-dev
True


In [ ]:
!wget -q https://essentia.upf.edu/models/autotagging/msd/msd-musicnn-1.pb
!wget -q https://essentia.upf.edu/models/autotagging/msd/msd-musicnn-1.json

In [ ]:
with open("msd-musicnn-1.json", "r") as f:
    metadata = json.load(f)

print(metadata.keys())
print("num classes:", len(metadata["classes"]))
print(metadata["classes"][:20])

dict_keys(['name', 'type', 'link', 'version', 'description', 'author', 'email', 'release_date', 'framework', 'framework_version', 'classes', 'model_types', 'dataset', 'schema', 'citation', 'inference'])
num classes: 50
['rock', 'pop', 'alternative', 'indie', 'electronic', 'female vocalists', 'dance', '00s', 'alternative rock', 'jazz', 'beautiful', 'metal', 'chillout', 'male vocalists', 'classic rock', 'soul', 'indie rock', 'Mellow', 'electronica', '80s']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
parquet_path = Path("/content/drive/MyDrive/data.parquet")
df = pd.read_parquet(parquet_path)

print(df.head())
print(df.columns.tolist())
print(df.shape)

                                               audio            title  \
0  {'bytes': b'ID3\x04\x00\x00\x00\x00\x02\x03TCO...             Food   
1  {'bytes': b'ID3\x04\x00\x00\x00\x00\x02=TIT2\x...     Electric Ave   
2  {'bytes': b'ID3\x04\x00\x00\x00\x00\x02\tTCON\...       This World   
3  {'bytes': b'ID3\x04\x00\x00\x00\x00\x05&TIT2\x...          Freeway   
4  {'bytes': b'ID3\x04\x00\x00\x00\x00\x05YTIT2\x...  Spiritual Level   

       artist  
0        AWOL  
1        AWOL  
2        AWOL  
3   Kurt Vile  
4  Nicky Cook  
['audio', 'title', 'artist']
(1230, 3)


In [ ]:
def resolve_audio_input(audio_value, parquet_path=None):
      if "bytes" in audio_value:
          audio_bytes = audio_value["bytes"]
          original_path = audio_value.get("path", "")

          suffix = ".bin"
          if isinstance(original_path, str) and "." in Path(original_path).name:
              suffix = Path(original_path).suffix or ".bin"

          tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
          tmp.write(audio_bytes)
          tmp.flush()
          tmp.close()

          return tmp.name

In [ ]:
row = df.iloc[0]

audio_path = resolve_audio_input(row["audio"], parquet_path)

print("audio_path:", audio_path)
print("artist:", row["artist"])
print("title:", row["title"])

audio_path: /tmp/tmp6rs_plq4.mp3
artist: AWOL
title: Food


In [ ]:
audio = es.MonoLoader(filename=audio_path, sampleRate=16000)()
print(audio.shape)
print(audio.dtype)

(479625,)
float32


In [ ]:
!pip -q install essentia-tensorflow

In [ ]:
model = es.TensorflowPredictMusiCNN(graphFilename="msd-musicnn-1.pb")
activations = model(audio)

print(type(activations))
print(np.array(activations).shape)

<class 'numpy.ndarray'>
(19, 50)


In [ ]:
mean_scores = np.mean(activations, axis=0)
top_idx = np.argsort(mean_scores)[::-1][:5]

top5 = []
for i in top_idx:
    top5.append({
        "label": metadata["classes"][i],
        "score": float(mean_scores[i])
    })

top5

[{'label': 'Hip-Hop', 'score': 0.875495195388794},
 {'label': 'rock', 'score': 0.05498085543513298},
 {'label': 'funk', 'score': 0.0360059030354023},
 {'label': 'soul', 'score': 0.03025987185537815},
 {'label': '90s', 'score': 0.029266420751810074}]

In [ ]:
import json
import numpy as np

def get_top5_features(audio_path, metadata, model):
    audio = es.MonoLoader(filename=audio_path, sampleRate=16000)()
    activations = model(audio)

    mean_scores = np.mean(activations, axis=0)
    top_idx = np.argsort(mean_scores)[::-1][:5]

    top5 = []
    for i in top_idx:
        top5.append({
            "label": metadata["classes"][i],
            "score": round(float(mean_scores[i]), 4)
        })
    return top5

In [ ]:
rows_json = []
temp_files = []

for idx, row in df.iterrows():
    try:
        audio_path = resolve_audio_input(row["audio"], parquet_path)
        temp_files.append(audio_path)

        top5 = get_top5_features(audio_path, metadata, model)

        rows_json.append(json.dumps(top5, ensure_ascii=False))
        if idx % 100 == 0:
          print('100 штук обработаны')

    except Exception as e:
        rows_json.append(json.dumps({
            "error": str(e)
        }, ensure_ascii=False))

100 штук обработаны
100 штук обработаны
100 штук обработаны
100 штук обработаны
100 штук обработаны
100 штук обработаны
100 штук обработаны
100 штук обработаны
100 штук обработаны
100 штук обработаны
100 штук обработаны
100 штук обработаны
100 штук обработаны


In [ ]:
df["music_features_json"] = rows_json
df[["artist", "title", "music_features_json"]].head()

,artist,title,music_features_json
0,AWOL,Food,"[{""label"": ""Hip-Hop"", ""score"": 0.8755}, {""labe..."
1,AWOL,Electric Ave,"[{""label"": ""Hip-Hop"", ""score"": 0.5082}, {""labe..."
2,AWOL,This World,"[{""label"": ""Hip-Hop"", ""score"": 0.9298}, {""labe..."
3,Kurt Vile,Freeway,"[{""label"": ""indie"", ""score"": 0.3608}, {""label""..."
4,Nicky Cook,Spiritual Level,"[{""label"": ""rock"", ""score"": 0.2631}, {""label"":..."


In [ ]:
df.to_parquet("/content/drive/MyDrive/data_with_music_features.parquet", index=False)
df[["artist", "title", "music_features_json"]].to_csv(
    "/content/drive/MyDrive/music_features_preview.csv",
    index=False
)